In [1]:
import pandas as pd
import subprocess
import scipy.stats as stats
import altair as alt
import numpy as np
from pathlib import Path
import sys
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

# ClinVar summary pre-preocessing

In [2]:
clinvar_file='/net/bbi/vol1//home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/variant_summary_2025-01.txt' #ClinVar Jan 2025 variant summary

In [3]:
original_clinvar_df = pd.read_csv(clinvar_file, sep='\t') #Read file

/tmp/16357859.1.shendure-login.q/ipykernel_1483878/1145797189.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  original_clinvar_df = pd.read_csv(clinvar_file, sep='\t') #Read file


In [4]:
clinvar_df = original_clinvar_df[['Type','Name', 'GeneSymbol', 'Assembly', 'Chromosome', 'Start', 'Stop', 'ClinicalSignificance', 'ReviewStatus', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']] #Grab desired columns
clinvar_df = clinvar_df.loc[(clinvar_df['Assembly']=='GRCh38') & (clinvar_df['Type']== 'single nucleotide variant') & (clinvar_df['Name'].str.contains(':'))].copy() #Filter for hg19 and SNVs only
filtered_full_clinvar_df = clinvar_df[clinvar_df['ReviewStatus'].isin(['criteria provided', 'multiple submitters, no conflicts', 'criteria provided, conflicting classifications', 'criteria provided, single submitter', 'reviewed by expert panel', 'practice guideline'])].copy() #Filter for 1-star plus

In [5]:
filtered_full_clinvar_df.head()

,Type,Name,GeneSymbol,Assembly,Chromosome,Start,Stop,ClinicalSignificance,ReviewStatus,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF
11,single nucleotide variant,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),NUBPL,GRCh38,14,31562125,31562125,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",31562125,G,A
17,single nucleotide variant,NM_000410.4(HFE):c.193A>T (p.Ser65Cys),HFE,GRCh38,6,26090957,26090957,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",26090957,A,T
19,single nucleotide variant,NM_000410.4(HFE):c.314T>C (p.Ile105Thr),HFE,GRCh38,6,26091078,26091078,Uncertain significance,"criteria provided, single submitter",26091078,T,C
21,single nucleotide variant,NM_000410.4(HFE):c.277G>C (p.Gly93Arg),HFE,GRCh38,6,26091041,26091041,Uncertain significance,"criteria provided, single submitter",26091041,G,C
23,single nucleotide variant,NM_000410.4(HFE):c.892+48G>A,HFE,GRCh38,6,26093008,26093008,Benign,"criteria provided, single submitter",26093008,G,A


In [6]:
filtered_full_clinvar_df['hgvs_c']=filtered_full_clinvar_df['Name'].transform(lambda x: x.split(':')[1]) #Pull hgvs_c from Name column
filtered_full_clinvar_df['aa_change']=filtered_full_clinvar_df['hgvs_c'].transform(lambda x: x.split(' ')[1][1:-1] if 'p.' in x else np.nan) #Gets amino acid change if applicable
filtered_full_clinvar_df['pos_id']=filtered_full_clinvar_df['GeneSymbol'] + ':' + filtered_full_clinvar_df['Start'].astype(str) + ':' + filtered_full_clinvar_df['AlternateAlleleVCF'] #Make variant ID field
filtered_full_clinvar_df['broad_consequence'] = filtered_full_clinvar_df['aa_change'].transform( 
    lambda x: 'non_coding' if (pd.isna(x) or 'p.' not in x)
              else 'synonymous' if '=' in x
              else 'non_synonymous'
) #broadly classsifies molecular consequence

filtered_full_clinvar_df['intronic_dist'] = filtered_full_clinvar_df['hgvs_c'].str.extract(r'[+-](\d+)') #Extracts distance from coding sequence
filtered_full_clinvar_df.head()

,Type,Name,GeneSymbol,Assembly,Chromosome,Start,Stop,ClinicalSignificance,ReviewStatus,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,hgvs_c,aa_change,pos_id,broad_consequence,intronic_dist
11,single nucleotide variant,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),NUBPL,GRCh38,14,31562125,31562125,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",31562125,G,A,c.166G>A (p.Gly56Arg),p.Gly56Arg,NUBPL:31562125:A,non_synonymous,NaN
17,single nucleotide variant,NM_000410.4(HFE):c.193A>T (p.Ser65Cys),HFE,GRCh38,6,26090957,26090957,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",26090957,A,T,c.193A>T (p.Ser65Cys),p.Ser65Cys,HFE:26090957:T,non_synonymous,NaN
19,single nucleotide variant,NM_000410.4(HFE):c.314T>C (p.Ile105Thr),HFE,GRCh38,6,26091078,26091078,Uncertain significance,"criteria provided, single submitter",26091078,T,C,c.314T>C (p.Ile105Thr),p.Ile105Thr,HFE:26091078:C,non_synonymous,NaN
21,single nucleotide variant,NM_000410.4(HFE):c.277G>C (p.Gly93Arg),HFE,GRCh38,6,26091041,26091041,Uncertain significance,"criteria provided, single submitter",26091041,G,C,c.277G>C (p.Gly93Arg),p.Gly93Arg,HFE:26091041:C,non_synonymous,NaN
23,single nucleotide variant,NM_000410.4(HFE):c.892+48G>A,HFE,GRCh38,6,26093008,26093008,Benign,"criteria provided, single submitter",26093008,G,A,c.892+48G>A,NaN,HFE:26093008:A,non_coding,48


In [7]:
syn_noncoding_clinvar_df = filtered_full_clinvar_df[filtered_full_clinvar_df['broad_consequence'].isin(['synonymous', 'non_coding'])].copy() #Gets generally synonymous and non-coding variants (includes intronic) mostly
syn_noncoding_clinvar_df = syn_noncoding_clinvar_df[syn_noncoding_clinvar_df['ClinicalSignificance'].isin(['Benign', 'Pathogenic','Likely pathogenic', 'Pathogenic; drug response', 'Likely pathogenic; drug response', 
                                                                                                           'Likely benign','Benign/Likely benign', 'risk factor','Pathogenic/Likely pathogenic'])].copy() #Filters out VUS

In [8]:
#Renames ClinVar consequences
syn_noncoding_clinvar_df.loc[syn_noncoding_clinvar_df['ClinicalSignificance'].str.contains('enign'), 'ClinicalSignificance'] = 'BLB'
syn_noncoding_clinvar_df.loc[(syn_noncoding_clinvar_df['ClinicalSignificance'].str.contains('hogenic')) | (syn_noncoding_clinvar_df['ClinicalSignificance'].str.contains('risk factor')), 'ClinicalSignificance'] = 'PLP'

for_vcf = syn_noncoding_clinvar_df.rename(columns = {'Chromosome': 'chrom', 
                                                     'PositionVCF': 'pos',
                                                     'ReferenceAlleleVCF': 'ref',
                                                     'AlternateAlleleVCF': 'alt'
                                                    }
                                         ) #Renames clinvar for VCF building

for_vcf = for_vcf[~for_vcf['chrom'].isin(['MT', 'Un'])] #Drops other chromosomes

#Some error checking and filtering for weird rows
for_vcf=for_vcf[for_vcf['pos']>0]
for_vcf=for_vcf[for_vcf['alt']!=for_vcf['ref']]
for_vcf = for_vcf.sort_values(["chrom", "pos"])

plp_vcf_df = for_vcf[for_vcf['ClinicalSignificance']=='PLP']

In [9]:
#Saves VCF if needed
save_vcf=False
if save_vcf:
    with open('/net/bbi/vol1/home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/20260429_clinvar_vcf.vcf', 'w') as f:
        f.write("##fileformat=VCFv4.3\n")
        f.write("##reference=GRCh38\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in for_vcf.iterrows():
            f.write(
                f"{row['chrom']}\t{row['pos']}\t.\t"
                f"{row['ref'].upper()}\t{row['alt'].upper()}\t.\tPASS\t.\n"
            )

# Analysis Start

In [10]:
vcf_path = '/net/bbi/vol1//home/ivanw314/work/20260428_SpliceAI_ClinVar_benchmark_supporting_files/20260429_clinvar_vep_combined.txt'
raw_vep_df = pd.read_csv(vcf_path, sep='\t')

raw_vep_df['pos']=raw_vep_df['Location'].transform(lambda x: int(x.split(':')[1].split('-')[0]))
raw_vep_df['pos_id']=raw_vep_df['SYMBOL'] + ':' + raw_vep_df['pos'].astype(str) + ':' +  raw_vep_df['Allele']
raw_vep_df['first_consequence']=raw_vep_df['Consequence'].transform(lambda x: x.split(',')[0])

raw_vep_df['maxSpliceAI'] = raw_vep_df[['SpliceAI_pred_DS_AG', 'SpliceAI_pred_DS_AL', 'SpliceAI_pred_DS_DG', 'SpliceAI_pred_DS_DL']].max(axis=1)

In [11]:
vep_df = raw_vep_df[~raw_vep_df['maxSpliceAI'].isin(['-'])].copy()
vep_df['splice_impact']=vep_df[['SpliceAI_pred_DS_AG', 'SpliceAI_pred_DS_AL', 'SpliceAI_pred_DS_DG', 'SpliceAI_pred_DS_DL']].idxmax(axis=1)
vep_df.loc[vep_df['maxSpliceAI'].astype(float) < 0.2, 'splice_impact'] = '< Threshold'

vep_df = vep_df.replace({'SpliceAI_pred_DS_AG': 'Acceptor Gain',
        'SpliceAI_pred_DS_AL': 'Acceptor Loss',
        'SpliceAI_pred_DS_DG': 'Donor Gain',
        'SpliceAI_pred_DS_DL': 'Donor Loss'})

vep_df['simple_splice_impact'] = '< Threshold'
vep_df.loc[vep_df['maxSpliceAI'].astype(float) >= 0.2, 'simple_splice_impact'] = 'Splice Impact Predicted'

vep_df = vep_df[['SYMBOL', 'Consequence', 'first_consequence', 'pos', 'pos_id', 'maxSpliceAI', 'splice_impact', 'simple_splice_impact']]

vep_df = vep_df.rename(columns={'SYMBOL':'GeneSymbol'})
vep_df.head()

,GeneSymbol,Consequence,first_consequence,pos,pos_id,maxSpliceAI,splice_impact,simple_splice_impact
1,SAMD11,synonymous_variant,synonymous_variant,925956,SAMD11:925956:T,0.03,< Threshold,< Threshold
2,SAMD11,synonymous_variant,synonymous_variant,925980,SAMD11:925980:T,0.11,< Threshold,< Threshold
3,SAMD11,synonymous_variant,synonymous_variant,925986,SAMD11:925986:T,0.01,< Threshold,< Threshold
4,SAMD11,synonymous_variant,synonymous_variant,926010,SAMD11:926010:T,0.00,< Threshold,< Threshold
5,SAMD11,intron_variant,intron_variant,926025,SAMD11:926025:A,0.00,< Threshold,< Threshold


In [12]:
vep_df_filtered = vep_df[(vep_df['Consequence'].str.contains('intron_variant')) | (vep_df['Consequence'].str.contains('synonymous'))] #Filters for any variant whose annotation contains intron or synonymous. This may include variants with splice region
vep_df_strict_filtered=vep_df[vep_df['first_consequence'].isin(['intron_variant', 'synonymous_variant'])] #Filters for pure intronic and synonymous variants only

dfs_for_analysis = {'all': vep_df_filtered,
                    'strict': vep_df_strict_filtered
                   }

## Analysis Helper Functions

In [64]:
def consequence_bars(df, filter_type):

    grouped = df.groupby(['first_consequence', 'ClinicalSignificance'])

    proportion_df = []
    
    for group, subset in grouped:
        group_df = pd.concat([subset.value_counts(subset=['first_consequence', 'ClinicalSignificance', 'simple_splice_impact']), subset.value_counts(subset=['first_consequence', 'ClinicalSignificance', 'simple_splice_impact'],normalize=True)], axis=1, keys=['count', 'proportion']).reset_index()
        proportion_df.append(group_df)

    proportion_df = pd.concat(proportion_df)
    #proportion_df = df.value_counts(subset=['first_consequence', 'ClinicalSignificance', 'simple_splice_impact'], normalize=True).reset_index()


    bar = alt.Chart(proportion_df).mark_bar().encode(
        x=alt.X('first_consequence:N'),
        y=alt.Y('proportion:Q'),
        color=alt.Color('simple_splice_impact:N')
    ).properties(width =300, height = 400).facet('ClinicalSignificance')
    
    bar.display()
    print(proportion_df)

## Batched Analysis

In [65]:
for key in dfs_for_analysis.keys():
    df_for_analysis = dfs_for_analysis[key]

    final_df = pd.merge(for_vcf, df_for_analysis, on=['pos_id', 'pos', 'GeneSymbol'], how='inner')[['Name', 'GeneSymbol', 'chrom', 'ClinicalSignificance', 'ReviewStatus', 'pos', 'ref', 'alt', 'hgvs_c', 'aa_change', 'pos_id', 'Consequence','first_consequence', 'maxSpliceAI', 'splice_impact', 'simple_splice_impact']]

    consequence_bars(final_df, key)

alt.FacetChart(...)

                     first_consequence ClinicalSignificance  \
0                       intron_variant                  BLB   
1                       intron_variant                  BLB   
0                       intron_variant                  PLP   
1                       intron_variant                  PLP   
0        splice_donor_5th_base_variant                  BLB   
1        splice_donor_5th_base_variant                  BLB   
0        splice_donor_5th_base_variant                  PLP   
1        splice_donor_5th_base_variant                  PLP   
0          splice_donor_region_variant                  BLB   
1          splice_donor_region_variant                  BLB   
0          splice_donor_region_variant                  PLP   
1          splice_donor_region_variant                  PLP   
0  splice_polypyrimidine_tract_variant                  BLB   
1  splice_polypyrimidine_tract_variant                  BLB   
0  splice_polypyrimidine_tract_variant                 

alt.FacetChart(...)

    first_consequence ClinicalSignificance     simple_splice_impact   count  \
0      intron_variant                  BLB              < Threshold  165789   
1      intron_variant                  BLB  Splice Impact Predicted    1461   
0      intron_variant                  PLP  Splice Impact Predicted     238   
1      intron_variant                  PLP              < Threshold     132   
0  synonymous_variant                  BLB              < Threshold  406869   
1  synonymous_variant                  BLB  Splice Impact Predicted    4575   
0  synonymous_variant                  PLP  Splice Impact Predicted      65   
1  synonymous_variant                  PLP              < Threshold      62   

   proportion  
0    0.991265  
1    0.008735  
0    0.643243  
1    0.356757  
0    0.988881  
1    0.011119  
0    0.511811  
1    0.488189  
